# Read the opsim database

In [ ]:
import opsimsummaryv2 as op
import os

## Configuration

In [ ]:
opsim_path = "/Users/dagoret/DATA/OpSim/baseline_v5.3.0_10yrs.db" 

In [ ]:
filename = os.path.basename(opsim_path)

## Read

In [ ]:
sql_engine = op.OpSimSurvey._get_sql_engine(opsim_path)

In [ ]:
df = op.OpSimSurvey._get_df_from_sql(sql_engine)

In [ ]:
df.columns

## Configuration

In [ ]:
BANDS_ORDER = ["ugrizy"]

In [ ]:
all_df = [ df[df["filter"] == b ] for b in BANDS_ORDER ]

## Analyse

In [ ]:
df["filter"] = df["filter"].str.extract(r'([a-zA-Z]+)')

In [ ]:
df["filter"]

In [ ]:
df

## Visualize each column vs MJD, colored by band

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.time import Time

In [ ]:
# Standard Rubin/LSST filter colors
FILTER_COLORS = {
    "u": "#56b4e9",
    "g": "#008060",
    "r": "#ff4000",
    "i": "#850000",
    "z": "#6600cc",
    "y": "#000000",
}
FILTER_ORDER = ["u", "g", "r", "i", "z", "y"]

In [ ]:
def get_year_ticks(mjd_min, mjd_max):
    """
    Compute the MJD of January 1st for every year covered by
    [mjd_min, mjd_max], used for axis ticks and axvlines.
    """
    t_min = Time(mjd_min, format="mjd").datetime
    t_max = Time(mjd_max, format="mjd").datetime

    year_mjds = []
    year_labels = []
    for year in range(t_min.year, t_max.year + 2):
        t_jan1 = Time(f"{year}-01-01T00:00:00", format="isot", scale="utc")
        if mjd_min <= t_jan1.mjd <= mjd_max:
            year_mjds.append(t_jan1.mjd)
            year_labels.append(f"{year}-01-01")
    return np.array(year_mjds), year_labels

In [ ]:
def plot_column_vs_mjd(df, column, mjd_column="observationStartMJD", filter_column="filter", figsize=(12, 5),simuname=filename):
    """
    Plot a dataframe column versus MJD, with points colored by
    Rubin/LSST band, a secondary top axis showing readable dates
    (YYYY-MM-DD), and vertical dashed lines marking the start of each year.
    """
    fig, ax = plt.subplots(figsize=figsize)

    mjd = df[mjd_column].values
    values = df[column]

    # Scatter per band so each band gets its own color and legend entry
    for band in FILTER_ORDER:
        mask = df[filter_column] == band
        if mask.sum() == 0:
            continue
        ax.scatter(
            mjd[mask.values],
            values[mask].values,
            s=3,
            alpha=0.5,
            color=FILTER_COLORS[band],
            label=band,
        )

    ax.set_xlabel("MJD")
    ax.set_ylabel(column)
    ax.set_title(f"{column} vs MJD ({simuname})")
    ax.legend(markerscale=5, title="band", loc="center left", fontsize=8,bbox_to_anchor=(1, 0.5))


    # Vertical lines at the start of each year
    mjd_min, mjd_max = np.nanmin(mjd), np.nanmax(mjd)
    year_mjds, year_labels = get_year_ticks(mjd_min, mjd_max)
    for ymjd in year_mjds:
        ax.axvline(ymjd, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)

    # Secondary top axis with readable dates
    ax_top = ax.twiny()
    ax_top.set_xlim(ax.get_xlim())
    ax_top.set_xticks(year_mjds)
    ax_top.set_xticklabels(year_labels, rotation=45, ha="left")
    ax_top.set_xlabel("Date")

    fig.tight_layout()
    return fig, ax

## Categorical columns: stacked horizontal bar plots by band

In [ ]:
def plot_category_stacked_barh(df, column, filter_column="filter", max_categories=30, figsize=(10, 8)):
    """
    Plot a horizontal bar chart of value counts for a categorical column,
    with each bar stacked by band (in FILTER_ORDER), using the standard
    Rubin/LSST filter colors.
    """
    counts = pd.crosstab(df[column], df[filter_column])
    bands = [b for b in FILTER_ORDER if b in counts.columns]
    counts = counts[bands]

    # Keep only the most populated categories for readability
    totals = counts.sum(axis=1).sort_values(ascending=False)
    counts = counts.loc[totals.index[:max_categories]]
    # Reverse so the largest category appears at the top of the plot
    counts = counts.iloc[::-1]

    fig, ax = plt.subplots(figsize=figsize)
    left = np.zeros(len(counts))
    for band in bands:
        ax.barh(
            counts.index.astype(str),
            counts[band].values,
            left=left,
            color=FILTER_COLORS[band],
            label=band,
        )
        left = left + counts[band].values

    ax.set_xlabel("Number of observations")
    ax.set_ylabel(column)
    ax.set_title(f"{column} by band ({filename})")
    ax.legend(title="band", loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)
    fig.tight_layout()
    return fig, ax

In [ ]:
# Columns not plotted: the ones already used to define the bands
CATEGORY_SKIP_COLUMNS = {"filter", "band"}

for column in df.columns:
    if column in CATEGORY_SKIP_COLUMNS:
        continue
    if pd.api.types.is_numeric_dtype(df[column]):
        continue
    fig, ax = plot_category_stacked_barh(df, column)
    plt.show()

In [ ]:
# Columns not plotted: the MJD itself, and non-numeric / categorical columns
SKIP_COLUMNS = {"observationStartMJD", "band", "filter"}

for column in df.columns:
    if column in SKIP_COLUMNS:
        continue
    if not pd.api.types.is_numeric_dtype(df[column]):
        continue
    fig, ax = plot_column_vs_mjd(df, column)
    plt.show()